# Notebook 36 — Live-channel analysis: does damage track surviving input channels?

Notebook 35 established a dose-response of collapse against surviving input-interface weights in both the
CNN (conv.0, 192 taps) and the MLP (input layer, 9,984 weights), with the MLP collapsing at the CNN's
surviving count. This notebook tests the quantity that should mediate the effect: a filter or unit whose
input weights are all zero is dead, so what the rest of the network sees is the number of **live input
channels**, not the number of surviving weights. For the MLP a second quantity applies: the number of
**covered input features** (features with at least one surviving weight to any unit); a CNN tap is shared
across positions, so any live filter covers every feature.

All quantities are read from the saved checkpoints of Notebooks 11, 34 and 35 (55 pruned models); no
model is trained or evaluated. Per-run macro-F1 comes from the committed Notebook 35 table. A random-mask
coverage model gives the expected live-channel count under uniform tap survival, so magnitude pruning's
tendency to concentrate survivors can be measured against it.

**Gate (stated before running).** (A) Spearman between live input channels and per-run macro-F1 is at
least 0.9 within each architecture (per-seed runs, not dose means). (B) A single live-channel threshold
separates collapsed runs (macro-F1 loss > 0.15) from intact runs across both architectures with at least
90% accuracy. CPU runtime is sufficient.

In [ ]:
# --- Colab bootstrap (CPU is enough) ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys, math, json as _json
os.chdir(REPO); sys.path.insert(0, REPO)
import numpy as np, pandas as pd, torch
from scipy.stats import spearmanr
from src.config import CFG, PATHS
from src.comnet_audit import environment_record, write_json
DATASET = 'ciciot2023'; SEEDS = list(CFG['seeds']); OUT = PATHS.tables('comnet')

CNN_RUNS = {0.0: 'layerwise80_protect_conv0_paired', 0.2: 'conv0dose20_paired', 0.5: 'conv0dose50_paired',
            0.8: 'prune80_paired', 0.9: 'conv0dose90_paired', 0.95: 'conv0dose95_paired'}
MLP_RUNS = {0.8: 'inputdose8000_mlp_paired', 0.9: 'inputdose9000_mlp_paired', 0.9808: 'inputdose9808_mlp_paired',
            0.9904: 'inputdose9904_mlp_paired', 0.9962: 'inputdose9962_mlp_paired'}
print('runs:', len(CNN_RUNS) * len(SEEDS) + len(MLP_RUNS) * len(SEEDS))

In [ ]:
# Live-channel and feature-coverage counts from saved masks
def input_stats(arch, cell, seed):
    ck = torch.load(PATHS.model(DATASET, arch, cell, seed), map_location='cpu', weights_only=False)
    sd = ck['state_dict'] if isinstance(ck, dict) and 'state_dict' in ck else ck
    if arch == 'cnn1d':
        w = sd['conv.0.weight']                      # (filters=64, in=1, k=3)
        nz = (w != 0)
        surviving = int(nz.sum()); live = int(nz.reshape(nz.shape[0], -1).any(dim=1).sum())
        covered = 39 if live > 0 else 0                  # shared taps: any live filter reads every feature position
        return {'surviving_input_weights': surviving, 'live_channels': live, 'channels_total': int(w.shape[0]), 'covered_features': int(covered)}
    w = sd['body.0.weight']                          # (units=256, features=39)
    nz = (w != 0)
    return {'surviving_input_weights': int(nz.sum()), 'live_channels': int(nz.any(dim=1).sum()), 'channels_total': int(w.shape[0]),
            'covered_features': int(nz.any(dim=0).sum())}

macro = pd.read_csv(OUT / 'input_starvation_macro_f1_wide.csv')
m0 = macro[macro.cell == 'M0'].set_index(['arch', 'seed'])['test_macro_f1']
rows = []
for arch, runs in (('cnn1d', CNN_RUNS), ('mlp', MLP_RUNS)):
    for dose, cell in runs.items():
        for seed in SEEDS:
            st = input_stats(arch, cell, seed)
            f1 = macro[(macro.arch == arch) & (macro.seed == seed) & (macro.cell == cell)].test_macro_f1
            assert len(f1) == 1, (arch, cell, seed)
            rows.append({'arch': arch, 'dose': dose, 'seed': seed, 'cell': cell, **st,
                         'test_macro_f1': float(f1.iloc[0]), 'macro_f1_loss': float(m0[(arch, seed)] - f1.iloc[0])})
per_run = pd.DataFrame(rows); per_run.to_csv(OUT / 'live_channel_per_run.csv', index=False)
summ = per_run.groupby(['arch', 'dose']).agg(surviving=('surviving_input_weights', 'mean'), live_channels=('live_channels', 'mean'),
        live_sd=('live_channels', 'std'), covered_features=('covered_features', 'mean'), macro_f1=('test_macro_f1', 'mean'),
        loss=('macro_f1_loss', 'mean')).reset_index()
print(summ.round(3).to_string(index=False))

In [ ]:
# Random-mask coverage model: expected live channels if T of N weights survived uniformly at random.
# P(channel with k weights has >=1 survivor) = 1 - C(N-k, T) / C(N, T), computed in log space.
def log_comb(n, k):
    if k < 0 or k > n: return -math.inf
    return math.lgamma(n + 1) - math.lgamma(k + 1) - math.lgamma(n - k + 1)
def expected_live(N, k, F, T):
    p_dead = math.exp(log_comb(N - k, T) - log_comb(N, T)) if T <= N - k else 0.0
    return F * (1 - p_dead)
cov = []
for arch, N, k, F in (('cnn1d', 192, 3, 64), ('mlp', 39 * 256, 39, 256)):
    for _, r in summ[summ.arch == arch].iterrows():
        T = int(round(r.surviving))
        row = {'arch': arch, 'dose': r.dose, 'surviving': T, 'observed_live': r.live_channels, 'expected_live_random': expected_live(N, k, F, T)}
        if arch == 'mlp': row['expected_covered_features_random'] = expected_live(N, 256, 39, T)
        cov.append(row)
cov = pd.DataFrame(cov); cov['observed_minus_expected'] = cov.observed_live - cov.expected_live_random
cov.to_csv(OUT / 'live_channel_coverage_model.csv', index=False)
print(cov.round(2).to_string(index=False))

In [ ]:
# Gate: within-architecture rank correlation on per-seed runs, and a single cross-architecture threshold
res = {}
for arch in ('cnn1d', 'mlp'):
    d = per_run[per_run.arch == arch]
    res[arch] = {'rho_live_vs_f1': spearmanr(d.live_channels, d.test_macro_f1).correlation,
                 'rho_weights_vs_f1': spearmanr(d.surviving_input_weights, d.test_macro_f1).correlation,
                 'rho_covered_vs_f1': spearmanr(d.covered_features, d.test_macro_f1).correlation if d.covered_features.nunique() > 1 else float('nan')}
    print(arch, {k: round(v, 3) for k, v in res[arch].items()})

per_run['collapsed'] = per_run.macro_f1_loss > 0.15
best = None
for thr in sorted(per_run.live_channels.unique()):
    pred = per_run.live_channels < thr
    acc = float((pred == per_run.collapsed).mean())
    if best is None or acc > best[1]: best = (int(thr), acc)
thr_live, acc_live = best
best_w = None
for thr in sorted(per_run.surviving_input_weights.unique()):
    acc = float(((per_run.surviving_input_weights < thr) == per_run.collapsed).mean())
    if best_w is None or acc > best_w[1]: best_w = (int(thr), acc)
print(f'\nsingle live-channel threshold: collapsed iff live < {thr_live}  -> accuracy {acc_live:.3f} over {len(per_run)} runs')
print(f'single surviving-weight threshold: collapsed iff weights < {best_w[0]} -> accuracy {best_w[1]:.3f}')
mis = per_run[(per_run.live_channels < thr_live) != per_run.collapsed][['arch', 'dose', 'seed', 'live_channels', 'macro_f1_loss']]
print('misclassified runs under the live-channel threshold:'); print(mis.to_string(index=False) if len(mis) else '  none')

verdict = pd.DataFrame([
 {'criterion': 'A_cnn_rho_live_vs_f1_ge_0.9', 'value': round(res['cnn1d']['rho_live_vs_f1'], 4), 'pass': bool(res['cnn1d']['rho_live_vs_f1'] >= 0.9)},
 {'criterion': 'A_mlp_rho_live_vs_f1_ge_0.9', 'value': round(res['mlp']['rho_live_vs_f1'], 4), 'pass': bool(res['mlp']['rho_live_vs_f1'] >= 0.9)},
 {'criterion': 'B_single_live_threshold_acc_ge_0.9', 'value': f'live<{thr_live}: acc {acc_live:.3f}', 'pass': bool(acc_live >= 0.9)},
 {'criterion': 'ref_single_weight_threshold_acc', 'value': f'weights<{best_w[0]}: acc {best_w[1]:.3f}', 'pass': ''},
])
print(); print(verdict.to_string(index=False))
verdict.to_csv(OUT / 'live_channel_gate_verdict.csv', index=False)
write_json(OUT / 'live_channel_environment.json', {'seeds': SEEDS, 'cnn_runs': CNN_RUNS, 'mlp_runs': MLP_RUNS, 'environment': environment_record()})

In [ ]:
# --- Commit + push: main only, own files only ---
import subprocess, shutil, glob
_b = subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip()
assert _b == 'main', f'checked-out branch is {_b!r}; run `git checkout main` first'
subprocess.run(['git', 'config', '--global', 'user.name', 'Md Anas Biswas'], check=True)
subprocess.run(['git', 'config', '--global', 'user.email', 'anasbiswas@gmail.com'], check=True)
cred = '/content/drive/MyDrive/IoT_Trust_Research/.git-credentials'
if os.path.exists(cred):
    shutil.copy(cred, '/root/.git-credentials'); subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
_own = 'notebooks/36_live_channel_analysis.ipynb'
if os.path.exists(_own):
    d = _json.load(open(_own))
    for c in d.get('cells', []):
        if c.get('cell_type') == 'code': c['outputs'] = []; c['execution_count'] = None
    _json.dump(d, open(_own, 'w'), indent=1)
subprocess.run(['git', 'add', _own] + glob.glob('results/tables/comnet/live_channel_*'), check=True)
r = subprocess.run(['git', 'commit', '-m', 'notebook 36: live-channel analysis of input starvation - per-run live channels vs macro-F1, random-mask coverage model, single cross-architecture threshold'], capture_output=True, text=True)
print(r.stdout or r.stderr)
print(subprocess.run(['git', 'push'], capture_output=True, text=True).stderr or 'pushed')
print(subprocess.run(['git', 'log', '--oneline', '-2'], capture_output=True, text=True).stdout)